In [ ]:
import os
import pandas as pd
import numpy as np
from glob import glob
import matplotlib.pyplot as plt

# CSV 경로
CSV_FOLDER = "/home/jyseo/hand_journal/data_MHAV/assemble/hg/assemble_hexNut-bolt_hand_hg/pose_2d_mp_csv"
csv_files = sorted(glob(os.path.join(CSV_FOLDER, "*.csv")))

def extract_right_hand(csv_path):
    df = pd.read_csv(csv_path)
    if len(df) == 42:
        return df.iloc[:21][["x", "y"]].values
    return None

rotation_angles = []

for i in range(1, len(csv_files)):
    kp_prev = extract_right_hand(csv_files[i - 1])
    kp_curr = extract_right_hand(csv_files[i])

    if kp_prev is not None and kp_curr is not None:
        # 손가락 방향 벡터 사용: 손목(0) → 중지 MCP(9)
        palm_prev = kp_prev[9] - kp_prev[0]
        palm_curr = kp_curr[9] - kp_curr[0]

        # 2D cross product로 회전 방향 및 각도 계산
        cross = palm_prev[0] * palm_curr[1] - palm_prev[1] * palm_curr[0]
        dot = np.dot(palm_prev, palm_curr)
        norm_product = np.linalg.norm(palm_prev) * np.linalg.norm(palm_curr)

        if norm_product > 0:
            angle = np.arctan2(cross, dot)
            angle_deg = np.degrees(angle)
            rotation_angles.append(angle_deg)

# 누적 회전 계산
total_rotation = np.sum(rotation_angles)

print("🌀 누적 회전량: {:.2f}도".format(total_rotation))
if total_rotation < -20:
    print("🔁 손이 몸 안쪽 방향 회전 (나사 풀기 가능성)")
elif total_rotation > 20:
    print("🔂 손이 몸 바깥 쪽 방향 회전 (나사 조이기 가능성)")
else:
    print("⏸ 회전이 거의 없음")

# 회전각 시각화
plt.figure(figsize=(10, 4))
plt.plot(rotation_angles, label="회전각 (프레임 간)")
plt.axhline(0, color='gray', linestyle='--')
plt.title("Right Hand Rotation Between Frames")
plt.xlabel("Frame Index")
plt.ylabel("Rotation (degree)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()
